# ETAPA FINAL: MODELADO Y EVALUACION

## 01-CONFIGURACION, CARGA Y TRANSFORMACION

### Importamos las librerías necesarias y aplicamos la transformación logarítmica al Target.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# Importaciones de modelos y herramientas
from sklearn.linear_model import Ridge # Baseline estable
from sklearn.svm import SVR # Modelo requerido
from sklearn.impute import SimpleImputer # CRÍTICO para el Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Carga y Transformación Inicial ---
try:
    ruta_dataset = r'G:\tecnicatura en cd e ia\2025\SEGUNDO AÑO\2do año 1er cuatrimestre\aprendizaje automatico\PROYECTOS AA\actividades de ML\PARCIAL_AA_2025\data\04-dataset definitivo\df_eda_final_para_ML.parquet'
    df = pd.read_parquet(ruta_dataset)
    print("✅ Datos principales cargados correctamente desde Parquet.")
    
except FileNotFoundError:
    print("❌ Error: Revisa la ruta a tu dataset limpio.")
    df = pd.DataFrame()
    
if not df.empty:
    # 1. TRASFORMACIONES LOGARÍTMICAS CLAVE (Si no están en el archivo Parquet)
    if 'Log_FOB_USD' not in df.columns and 'FOB(USD)' in df.columns:
        df['Log_FOB_USD'] = np.log1p(df['FOB(USD)'])
    if 'Log_Peso_Neto_kg' not in df.columns and 'Peso_Neto_kg' in df.columns:
        df['Log_Peso_Neto_kg'] = np.log1p(df['Peso_Neto_kg'])
        
    # 2. DEFINICIÓN DE FEATURES Y TARGET (Basado en EDA)
    # Excluimos: Price_Unitario_USD_kg (baja correlación), variables sin transformar, Fecha y Año.
    FEATURE_LIST = [
        'Log_FOB_USD',
        'Log_Peso_Neto_kg',
        'Costo_Logistico_Pct',
        'mes',
        'Pais_Encoded_Target',
        'NCM_Encoded_Target',
    ]

    X = df[FEATURE_LIST] 
    y = np.log1p(df['CIF(USD)']) # Reaseguramos la creación del Target transformado
    
   # --- 3. División en entrenamiento y prueba y MUESTREO (CRÍTICO para SVR) ---
# Usamos el 80% para entrenamiento y el 20% para prueba del dataset original.
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Aplicamos muestreo para reducir el tamaño del dataset de entrenamiento y prueba
# Muestreamos el 10% de los datos para que el SVR pueda ejecutarse.
SAMPLE_FRACTION = 0.10 

# Muestreo del conjunto de Entrenamiento
X_train = X_train_full.sample(frac=SAMPLE_FRACTION, random_state=42)
y_train = y_train_full.sample(frac=SAMPLE_FRACTION, random_state=42)

# Muestreo del conjunto de Prueba
X_test = X_test_full.sample(frac=SAMPLE_FRACTION, random_state=42)
y_test = y_test_full.sample(frac=SAMPLE_FRACTION, random_state=42)


print(f"Datos Originales: {len(X)} instancias")
print(f"Datos de Entrenamiento (Muestreados al {SAMPLE_FRACTION*100}%): {len(X_train)} instancias")
print(f"Datos de Prueba (Muestreados al {SAMPLE_FRACTION*100}%): {len(X_test)} instancias")
# Esto debería darte aproximadamente 28,000 en Train y 7,000 en Test

✅ Datos principales cargados correctamente desde Parquet.
Datos Originales: 347874 instancias
Datos de Entrenamiento (Muestreados al 10.0%): 27830 instancias
Datos de Prueba (Muestreados al 10.0%): 6958 instancias


## 2-INGENIERIA DE FEATURES Y PIPELINE

### Definimos las listas de features numéricas.

In [4]:
print("\n--- 2. Preprocesador con Imputación y Escalado (Pipeline) ---")

# Todas las features seleccionadas son numéricas y requieren preprocesamiento
features_to_preprocess = FEATURE_LIST

# Definición del Pipeline Numérico: Imputación (maneja NaN/Inf) + Escalado (CRÍTICO para Ridge/SVR)
numerical_transformer = Pipeline(steps=[
    # 1. Imputación: Esencial para manejar los NaN que quedaron de los infinitos
    ('imputer', SimpleImputer(strategy='mean')), 
    # 2. Escalado: CRÍTICO para Ridge y SVR. Centra y reduce la varianza.
    ('scaler', StandardScaler())
])

# Definición del ColumnTransformer: Aplica el pipeline numérico a todas las features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, features_to_preprocess)
    ],
    remainder='drop' # Descartar otras columnas
)

print("Preprocesador (ColumnTransformer) definido con Imputación y StandardScaler.")


--- 2. Preprocesador con Imputación y Escalado (Pipeline) ---
Preprocesador (ColumnTransformer) definido con Imputación y StandardScaler.


## 3-ENTRENAMIENTO Y EVALUACION DE MODELOS DE REGRESION

### Ejecución de los Modelos (Ridge y SVR)

In [5]:
# A. Función de Evaluación Final (Revertir Logaritmo)
LOG_PRED_MAX = 70 # Límite de seguridad para evitar overflow en np.expm1

def evaluate_model(model, X_test, y_test):
    # 1. Predicción del logaritmo del Target
    y_pred_log = model.predict(X_test)
    
    # 2. CLIPPING: Limitar las predicciones logarítmicas a un valor seguro
    y_pred_log = np.clip(y_pred_log, a_min=None, a_max=LOG_PRED_MAX) 
    
    # 3. INVERSIÓN DE LA TRANSFORMACIÓN (Ahora sin overflow)
    y_test_original = np.expm1(y_test)
    y_pred_original = np.expm1(y_pred_log)
    
    # 4. Calcular Métricas de Regresión
    mae = mean_absolute_error(y_test_original, y_pred_original)
    mse = mean_squared_error(y_test_original, y_pred_original)
    r2 = r2_score(y_test_original, y_pred_original)
    
    return mae, np.sqrt(mse), r2 # Devolvemos MAE, RMSE, y R2

# B. Entrenamiento y Resultados

# 1. Ridge Regressor (Baseline Estabilizado)
print("\n--- Entrenando: Ridge Regressor (Regresión Lineal Baseline) ---")
ridge_pipe = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', Ridge(random_state=42))])
ridge_pipe.fit(X_train, y_train)

mae_r, rmse_r, r2_r = evaluate_model(ridge_pipe, X_test, y_test)
print(f"  Resultados Ridge: MAE={mae_r:,.2f} | RMSE={rmse_r:,.2f} | R2={r2_r:.4f}")

# 2. SVR (Support Vector Regressor - Modelo Clave)
print("\n--- Entrenando: SVR (Support Vector Regressor) ---")
print("✅ SVR corriendo en muestra reducida. Esto tomará varios minutos...")
svr_pipe = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', SVR(kernel='linear'))])
svr_pipe.fit(X_train, y_train) 

mae_s, rmse_s, r2_s = evaluate_model(svr_pipe, X_test, y_test)
print(f"  Resultados SVR: MAE={mae_s:,.2f} | RMSE={rmse_s:,.2f} | R2={r2_s:.4f}")


--- Entrenando: Ridge Regressor (Regresión Lineal Baseline) ---
  Resultados Ridge: MAE=9,688.23 | RMSE=136,644.52 | R2=0.9879

--- Entrenando: SVR (Support Vector Regressor) ---
✅ SVR corriendo en muestra reducida. Esto tomará varios minutos...
  Resultados SVR: MAE=11,553.87 | RMSE=83,225.70 | R2=0.9955


# 4. CONCLUSION FINAL Y COMPARACION DE MODELOS

## La etapa de Modelado y Evaluación concluyó con la comparación de un modelo lineal con regularización (Ridge) y un modelo de frontera avanzado (SVR), ambos ejecutados en el dataset transformado logarítmicamente y muestreado.

## A. Tabla Comparativa de Rendimiento

### Los resultados obtenidos en el conjunto de prueba son:

| Modelo de Regresión | MAE (USD) | RMSE (USD) | R2     | Conclusión                                  |
|---------------------|-----------|------------|--------|---------------------------------------------|
| Ridge Regressor     | 9,688.23  | 136,644.52 | 0.9879 | Baseline Sólido. Error bajo, alta varianza. |
| SVR (Linear Kernel) | 11,553.87 | 83,225.70  | 0.9955 | Ganador. Mayor precisión y menor error.     |



## B. Análisis y conclusión del trabajo

### Selección del mejor modelo

    El modelo Support Vector Regressor (SVR) se selecciona como el modelo ganador para la predicción del valor de importación (Valor_CIF_USD).

### Hallazgos Clave:

    Alta Precisión (R2): Ambos modelos lograron una precisión notable, lo cual valida las decisiones tomadas en el EDA, especialmente la transformación logarítmica de las variables de valor (Log_Valor_CIF_USD, Log_FOB_USD, Log_Peso_Neto_kg) y el uso del Target Encoding para las variables categóricas.

### Métrica de Precisión (R2): El SVR logró un coeficiente de determinación de R2=0.9955. Esto indica que el modelo explica el 99.55% de la varianza en el valor de la importación.

### Métrica de Error (RMSE): El SVR mostró un RMSE significativamente menor (83,225.70) en comparación con el Ridge (136,644.52).

El RMSE (Error Cuadrático Medio) castiga más fuertemente los errores grandes (outliers). El SVR, por su naturaleza, es más robusto y maneja las fronteras de decisión de forma más eficiente, lo que resulta en una menor varianza y un error promedio de predicción más bajo.

### Métrica MAE: El Ridge tuvo un MAE (Error Absoluto Medio) ligeramente menor. Sin embargo, en el contexto de regresión, el RMSE es la métrica de referencia para medir la calidad del modelo global.

# Conclusión Final

## Se confirma la hipótesis inicial: el valor de importación (Log_Valor_CIF_USD) puede ser predicho con alta precisión utilizando una combinación de variables transformadas logarítmicamente y features categóricas codificadas. El modelo SVR ofrece la mejor combinación de precisión y estabilidad de error, haciendo la predicción más confiable para el negocio.